# **1. Mount Google Drive**


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **2. Mengimport Library**

In [12]:
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
import os

# **3. Menentukan Path Folder dan Properti Fitur**

In [13]:
# list properti yang akan diekstraksi
maturity_properties = ['hue_mean', 'saturation_mean', 'value_mean', 'hue_std', 'saturation_std', 'value_std', 'texture_contrast']

# path folder yang berisi gambar
folder_path = '/content/drive/MyDrive/ComputerVision/EkstraksiFitur/Gambar Kopi'
data = []

# **4. Loop Load Data file Citra dan Proses Ekstraksi Fitur**

In [14]:
# loop untuk setiap file citra di folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.jpg') or file_name.endswith('.jpeg') or file_name.endswith('.png') or file_name.endswith('.bmp'):
        row_data = []
        file_path = os.path.join(folder_path, file_name)
        row_data.append(file_name)

        # preprocessing gambar
        src = cv2.imread(file_path)
        src = cv2.resize(src, (300, 300))
        hsv = cv2.cvtColor(src, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)

        # ekstraksi properti warna
        hue_mean = np.mean(h)
        saturation_mean = np.mean(s)
        value_mean = np.mean(v)
        hue_std = np.std(h)
        saturation_std = np.std(s)
        value_std = np.std(v)

        # ekstraksi properti bentuk
        contours, hierarchy = cv2.findContours(cv2.Canny(src, 100, 200), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cnt = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(cnt, True)
        area = cv2.contourArea(cnt)
        circularity = 4 * np.pi * (area / perimeter ** 2)
        aspect_ratio = float(src.shape[0]) / src.shape[1]
        extent = area / (src.shape[0] * src.shape[1])
        solidity = area / cv2.contourArea(cv2.convexHull(cnt))

        # hitung kontras tekstur
        g = graycomatrix(v, distances=[1], angles=[0], symmetric=True, normed=True)
        contrast = graycoprops(g, 'contrast')
        # tambahkan ke data
        row_data.extend([hue_mean, saturation_mean, value_mean, hue_std, saturation_std, value_std, contrast])
        data.append(row_data)

# **5. Export Data Hasil Ekstraksi Fitur ke File Excel**

In [15]:
# export ke file Excel
df = pd.DataFrame(data, columns=['file_name'] + maturity_properties)
file_path = os.path.join(folder_path, '2318059EkstratsiFiturPhyton.xlsx')
df.to_excel(file_path, index=False)